# Research 3 — official Qwen3.8-27B application pilot

A100 + High-RAM. Model tools affect fictional memory only; no real filesystem, shell or network tools. No cgroup prerequisite. Trusted code reads weights and writes artifacts. Authorization: 50 units from initial observed balance 80, less prior usage; 2-unit reserve. Run setup/download/feasibility first. The campaign has a 6-hour total notebook budget. Export privately only after all model work stops; disconnect/delete the runtime when done.


In [ ]:
import base64, gzip, hashlib, json, os, subprocess, sys, time, uuid
from pathlib import Path
SESSION_STARTED = time.monotonic()
RUN_ID = "qwen38_simple_v1_" + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid.uuid4().hex[:8]
ROOT = Path("/content/sp_lense_work") / RUN_ID
ROOT.mkdir(parents=True, exist_ok=False)
for name in ("checkpoints/adapters", "checkpoints/resume", "training/configs", "training/logs", "training/receipts", "evaluation/scenarios", "evaluation/trajectories", "evaluation/tool_calls", "evaluation/results", "activations", "reports", "code"):
    (ROOT / name).mkdir(parents=True, exist_ok=False)
PACKED_SOURCE = (
'...'
)
SOURCE_FILES = json.loads(gzip.decompress(base64.b64decode(PACKED_SOURCE)).decode())
for name, content in SOURCE_FILES.items():
    path = ROOT / "code" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8", newline="\n")
MODEL_PIN = {'repository': 'Qwen/Qwen3.8-27B', 'revision': '1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0', 'queried_utc': '2026-09-30T01:40:09.0713707Z', 'source': 'https://huggingface.co/api/models/Qwen/Qwen3.8-27B/revision/1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0?blobs=true', 'files': [{'name': '.gitattributes', 'size': 1570, 'sha256': None, 'git_blob_id': '52373fe24473b1aa44333d318f578ae6bf04b49b'}, {'name': 'LICENSE', 'size': 11544, 'sha256': None, 'git_blob_id': 'f938136e3adacfd92be087f6e113b5d6d97f678f'}, {'name': 'README.md', 'size': 65012, 'sha256': None, 'git_blob_id': 'bc8aa0e396cd029c21cd773cca21830d0ded28ec'}, {'name': 'chat_template.jinja', 'size': 8952, 'sha256': None, 'git_blob_id': 'c0c686f9c38d70d179fb7b5f5aa7530bc913dda3'}, {'name': 'config.json', 'size': 4312, 'sha256': None, 'git_blob_id': '706cebd746c4b6f2b1d1f892630867acfdfd3df8'}, {'name': 'crc32.txt', 'size': 238, 'sha256': None, 'git_blob_id': '6de5ee6a0c6596744baee911af0a5cdcb8d99a1e'}, {'name': 'generation_config.json', 'size': 202, 'sha256': None, 'git_blob_id': '023756cfadf88e5bf69eefeee3e172f38c448d64'}, {'name': 'merges.txt', 'size': 3353259, 'sha256': None, 'git_blob_id': 'a494e019ca1502219fd0128658b979e5f05ae8e8'}, {'name': 'model-00001-of-00018.safetensors', 'size': 3966730552, 'sha256': 'ba0ce20aae489ad196733da5064bcdf159a1fe84f53336648196e1ebb7751b1c', 'git_blob_id': '79b73cdd69afeca931d181f5d1e21ad2d73bb7b7'}, {'name': 'model-00002-of-00018.safetensors', 'size': 3043080328, 'sha256': '06a148c01bfbe3faa14a5f184a7ff29a706f7ae1c8b2705d2058e26d17a001fb', 'git_blob_id': '81cf43545884a227a01351942dfbc72577812316'}, {'name': 'model-00003-of-00018.safetensors', 'size': 2542796952, 'sha256': '2e1bf62cbcd406eaa64b60d10353e1f0ef4039d0976e56f05cabe953454f9968', 'git_blob_id': 'a2da5e6cd6307a6cae13d04820ab0d31b47b48bd'}, {'name': 'model-00004-of-00018.safetensors', 'size': 3988973152, 'sha256': '511e34063187882659753c4d93f3859f93c019fd438d8813071921c81d9a3f1a', 'git_blob_id': '175ddab3f2488ad181f73d789d2d778cf6d4fca6'}, {'name': 'model-00005-of-00018.safetensors', 'size': 2099339864, 'sha256': '635cb53446dc74f219740fc59e18b774f877b803b9722e289ca62575a6efa701', 'git_blob_id': '64890208fdd563c3aac0c2f55f69948bcaa1a492'}, {'name': 'model-00006-of-00018.safetensors', 'size': 3979553696, 'sha256': '0bc5214fac607f0e6cc92eec3789d4b8559410ef9fce66621ba8158e8410dae0', 'git_blob_id': '3ae29b86e4c773babfe17124dbd10d54d37819ec'}, {'name': 'model-00007-of-00018.safetensors', 'size': 2108759344, 'sha256': '80b0c49033e9a0d5762562aa12f4acdb7f54da586f3d0110f28c48d91cf07892', 'git_blob_id': 'b7e9c0f038884839f3dad38fef7e75ee3362c24f'}, {'name': 'model-00008-of-00018.safetensors', 'size': 3979553696, 'sha256': '7192c5b66185d3592927daabee1cc19e6f6e0ce75988ee20e824b624765fda79', 'git_blob_id': '0b58cbc52714e1c683711be87e7c2e33fb10e7a0'}, {'name': 'model-00009-of-00018.safetensors', 'size': 2108759344, 'sha256': 'af3c48cc37af44f3db6ae0579baf019180d48d9c527caa0a1f03ff85813a56d8', 'git_blob_id': '4fc0c6b28004d809225d0e115f30e9e11c2345dc'}, {'name': 'model-00010-of-00018.safetensors', 'size': 3979553696, 'sha256': '163490a76f3bea3a40855b7efc04ce6d27afaf1a34f0bbde495b9491f76457c9', 'git_blob_id': '9ec693a17c788084116c51b4f34632670df77612'}, {'name': 'model-00011-of-00018.safetensors', 'size': 2108759344, 'sha256': '5f3ae1b948aeee39da77aec558e8236cd65fe4d7cb7686a76bb007acc563c6d8', 'git_blob_id': '3c63baaed16d833432a919647af53ac76bb6b007'}, {'name': 'model-00012-of-00018.safetensors', 'size': 3979553696, 'sha256': 'a3de1c7114677a8f5ac5c4892c90e8238ea5c1e2038c80e757dfc87c3902ca55', 'git_blob_id': '8568b440a4666ed1b4cda0b5d21496378304f326'}, {'name': 'model-00013-of-00018.safetensors', 'size': 2108759344, 'sha256': '06ab79a41f74c9c5cb734816feb0c7fc364104b227165ee7391231e1155aa02a', 'git_blob_id': '724083995a2257194321a3d00b36fd7d4bb7348d'}, {'name': 'model-00014-of-00018.safetensors', 'size': 3979553696, 'sha256': '4138ed94603065ba884bbcadedb04d7718bb40117e85e6f5c6fc5b9c05b7a85b', 'git_blob_id': 'a60b80b6510d7938d951fbddc76aaadd121c6268'}, {'name': 'model-00015-of-00018.safetensors', 'size': 2108759344, 'sha256': '69224e27b9de4e7dbf6fc936c6eaae08447bda3b80a6c31a871ab451173afd22', 'git_blob_id': '7d4c34f3d716e1a47acff2f576b65b41a3ec4f3f'}, {'name': 'model-00016-of-00018.safetensors', 'size': 3979564040, 'sha256': '73cb9a1089fb6155cb648609478d6633be8a5c7d9ca5a05bc8925ce8a553cefe', 'git_blob_id': 'c67e3e42c9715298fa7f68c6b378bfb1d74290d0'}, {'name': 'model-00017-of-00018.safetensors', 'size': 2108759344, 'sha256': 'beb51f01056142ac4984bd800507b0dd0fd18de57f8e9ef6ea41d1a3598983a8', 'git_blob_id': '23dd899179729a4370c655e2c2241cfdbe4ef4ba'}, {'name': 'model-00018-of-00018.safetensors', 'size': 3392197344, 'sha256': '1d3479509e21494658f9b64d317f5ea8e55c4025d28c702d6c4d0b356ce8ea06', 'git_blob_id': 'a92b4073fc6a58ffc9334ac63360fc8d9cc50d33'}, {'name': 'model.safetensors.index.json', 'size': 112216, 'sha256': None, 'git_blob_id': 'da35e3c564457dface7d138f0b6cac284ff8958c'}, {'name': 'preprocessor_config.json', 'size': 390, 'sha256': None, 'git_blob_id': '2ea84a437d448ff71b08df68fdd949d5cc4ebb64'}, {'name': 'tokenizer.json', 'size': 12809320, 'sha256': '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3', 'git_blob_id': '9328ce9c41e80f6dd7bc2c66d8ae1fc93bf87440'}, {'name': 'tokenizer_config.json', 'size': 17928, 'sha256': None, 'git_blob_id': '5de744b3fca2129d7186979ae47c06be33903243'}, {'name': 'video_preprocessor_config.json', 'size': 385, 'sha256': None, 'git_blob_id': '3ba673a5ad7d4d13f54155ecd38b2a94a6dac8fe'}, {'name': 'vocab.json', 'size': 6722759, 'sha256': None, 'git_blob_id': '0aa0ce0658d60ac4a5d609f4eadb0e8e43514176'}]}
(ROOT / "training/configs/model_pin.json").write_text(json.dumps(MODEL_PIN, indent=2))
(ROOT / "manifest.json").write_text(json.dumps({"run_id":RUN_ID, "model":MODEL_PIN["repository"], "revision":MODEL_PIN["revision"], "boundary":"application-only fictional in-memory tools", "starting_balance_for_authorization":80, "authorized_units":50, "reserved_units":2, "thinking":False, "source_freeze":json.loads(SOURCE_FILES["data/FREEZE.json"])}, indent=2))
print("RUN_ROOT", ROOT)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout)
subprocess.run([sys.executable,"-m","pip","install","--quiet","--no-input","-r", str(ROOT / "code/requirements.txt")], check=True, timeout=900)
print("PINNED_PACKAGES_INSTALLED")


In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = ROOT / "model"
snapshot_download(repo_id=MODEL_PIN["repository"], revision=MODEL_PIN["revision"], local_dir=MODEL_DIR, token=False, max_workers=8, allow_patterns=["*.safetensors","*.json","*.jinja","merges.txt","vocab.json"])
MODEL_MANIFEST = {}
for item in MODEL_PIN["files"]:
    path = MODEL_DIR / item["name"]
    if not path.is_file():
        continue
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(16*1024**2), b""):
            digest.update(block)
    actual = digest.hexdigest()
    if item["sha256"] and actual != item["sha256"]:
        raise RuntimeError("Pinned model hash mismatch: " + item["name"])
    MODEL_MANIFEST[item["name"]] = {"bytes":path.stat().st_size,"sha256":actual}
assert len([name for name in MODEL_MANIFEST if name.endswith(".safetensors")]) == 18
(ROOT / "training/configs/model_manifest.json").write_text(json.dumps(MODEL_MANIFEST, indent=2))
print("OFFICIAL_MODEL_DOWNLOAD_VERIFIED", len(MODEL_MANIFEST), "files")


In [ ]:
sys.path.insert(0, str(ROOT / "code"))
from controller import run
PILOT_RECEIPT = run(ROOT, "feasibility", min(1800, 21600 - (time.monotonic()-SESSION_STARTED)))
FEASIBILITY = json.loads((ROOT / "training/receipts/feasibility.json").read_text())
print("FEASIBILITY_SUMMARY", json.dumps({key:value for key,value in FEASIBILITY.items() if key not in ("targets","generation")}, indent=2))


In [ ]:
# Conservatively admit the complete matched campaign from measured 1024-token updates.
remaining_seconds = 21600 - (time.monotonic()-SESSION_STARTED)
projected_training = max(FEASIBILITY["optimizer_block_seconds"]) * (214*3)
generation = FEASIBILITY["generation"]
seconds_per_token = generation["seconds"] / max(1, len(generation["token_ids"]))
# Conservative generation projection; caps are maxima, actual episodes can be shorter.
projected_evaluation = seconds_per_token * (4*(56*80 + 24*160 + 16*300))
projection = 1.25*(projected_training + projected_evaluation) + 600
print("CAMPAIGN_ADMISSION", json.dumps({"projected_seconds":projection,"remaining_seconds":remaining_seconds}))
if projection > remaining_seconds:
    raise RuntimeError("Measured campaign projection exceeds this bounded budget; retain pilot and adjust scope before full run.")
CAMPAIGN_RECEIPT = run(ROOT, "campaign", remaining_seconds)


## Private export after workers exit

Mount Drive only in this trusted phase. No base model weights are copied. Do not run the export cell concurrently with model work.


In [ ]:
# Run only after the model subprocess has exited; no workers while Drive is mounted.
import shutil, stat
from google.colab import drive
EXPORT_FILES = []
for path in ROOT.rglob("*"):
    relative = path.relative_to(ROOT)
    if relative.parts[0] in ("model","worker_home"):
        continue
    info = path.lstat()
    if stat.S_ISLNK(info.st_mode):
        raise RuntimeError("Export symlink rejected")
    if not stat.S_ISREG(info.st_mode):
        continue
    if info.st_size > 2*1024**3:
        raise RuntimeError("Oversized export rejected")
    EXPORT_FILES.append((path, relative, info.st_size))
drive.mount("/content/drive")
DEST = Path("/content/drive/MyDrive/sp_lense/research3/runs") / RUN_ID
DEST.mkdir(parents=True, exist_ok=False)
if shutil.disk_usage(DEST).free < sum(size for _,_,size in EXPORT_FILES) + 1024**3:
    raise RuntimeError("Insufficient Drive free space")
EXPORT_HASHES = {}
for source, relative, size in EXPORT_FILES:
    target = DEST / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
    def file_hash(path):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(16*1024**2), b""):
                digest.update(block)
        return digest.hexdigest()
    actual = file_hash(source)
    if actual != file_hash(target):
        raise RuntimeError("Drive copy hash mismatch")
    EXPORT_HASHES[str(relative)] = actual
(DEST / "EXPORT_HASHES.json").write_text(json.dumps(EXPORT_HASHES, indent=2))
drive.flush_and_unmount()
print("PRIVATE_EXPORT_VERIFIED", DEST, len(EXPORT_HASHES), "files")
